In [1]:
!pip install nltk
!pip install numpy
!pip install pandas
!pip install cvxopt
!pip install matplotlib
!pip install prettytable


In [2]:
import zipfile
import os

# Define the path to the zip file and the directory to extract to
zip_path = 'emails.zip'
extract_dir = 'emails/'

# Create the directory if it doesn't exist
if not os.path.exists(extract_dir):
    os.makedirs(extract_dir)

# Extract the zip file
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(f"Extraction complete. Files are extracted to '{extract_dir}'")


Extraction complete. Files are extracted to 'emails/'


In [3]:
import os
import nltk
import time
import string
import operator
from nltk.corpus import stopwords
from nltk.stem.wordnet import WordNetLemmatizer

# Download required NLTK data files if not already available
nltk.download('stopwords')
nltk.download('wordnet')

# Function to clean up the text
def text_cleanup(text):
    # Remove punctuation
    text_no_punct = ''.join([c for c in text if c not in string.punctuation])
    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    words = text_no_punct.split()
    words_filtered = [word.lower() for word in words if word.lower() not in stop_words]
    return words_filtered

start_time = time.time()

lmtzr = WordNetLemmatizer()
word_count = {}
processed_files = 0

# Update this path if your structure changes after extraction
directory_path = os.path.join("emails", "emails")

if not os.path.exists(directory_path):
    print(f"Directory '{directory_path}' not found.")
    exit()

# Iterate through files
for filename in os.listdir(directory_path):
    filepath = os.path.join(directory_path, filename)
    if os.path.isfile(filepath):
        with open(filepath, "r", encoding="utf-8", errors="ignore") as file:
            words = text_cleanup(file.read())
            for word in words:
                if not word.isdigit() and len(word) > 2:
                    lemma = lmtzr.lemmatize(word)
                    word_count[lemma] = word_count.get(lemma, 0) + 1

        processed_files += 1
        if processed_files % 100 == 0:
            print(f"Processed {processed_files} files.")

# Sort words by frequency
sorted_word_count = sorted(word_count.items(), key=operator.itemgetter(1), reverse=True)

# Write frequent words (count >= 100) to CSV
with open("wordslist.csv", "w", encoding="utf-8") as out_file:
    out_file.write("word,count\n")
    for word, freq in sorted_word_count:
        if freq < 100:
            break
        out_file.write(f"{word},{freq}\n")

print(f"\nDone. Total files processed: {processed_files}")
print(f"Time taken: {round(time.time() - start_time, 2)} seconds")


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


Processed 100 files.
Processed 200 files.
Processed 300 files.
Processed 400 files.
Processed 500 files.
Processed 600 files.
Processed 700 files.
Processed 800 files.
Processed 900 files.
Processed 1000 files.
Processed 1100 files.
Processed 1200 files.
Processed 1300 files.
Processed 1400 files.
Processed 1500 files.
Processed 1600 files.
Processed 1700 files.
Processed 1800 files.
Processed 1900 files.
Processed 2000 files.
Processed 2100 files.
Processed 2200 files.
Processed 2300 files.
Processed 2400 files.
Processed 2500 files.
Processed 2600 files.
Processed 2700 files.
Processed 2800 files.
Processed 2900 files.
Processed 3000 files.
Processed 3100 files.
Processed 3200 files.
Processed 3300 files.
Processed 3400 files.
Processed 3500 files.
Processed 3600 files.
Processed 3700 files.
Processed 3800 files.
Processed 3900 files.
Processed 4000 files.
Processed 4100 files.
Processed 4200 files.
Processed 4300 files.
Processed 4400 files.
Processed 4500 files.
Processed 4600 file

In [4]:
import os
import string
import numpy as np
import pandas as pd
from time import time
from nltk.corpus import stopwords
from nltk.stem.wordnet import WordNetLemmatizer

# Start timer
start_time = time()

# Load words from previously created wordslist.csv
df = pd.read_csv('wordslist.csv')
words = df['word']
word_list = words.tolist()  # Convert to regular list for efficiency
# Convert all elements in word_list to strings
word_list = [str(word) for word in word_list] #This line is added to convert all elements in word_list to strings using list comprehension.


# Initialize lemmatizer and stopwords
lmtzr = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# Define directory containing emails
directory = "emails/emails"  # Update if necessary

# Prepare the output CSV with header
with open("frequency.csv", "w", encoding='utf-8') as f_out:
    f_out.write(','.join(word_list) + ',output\n')

# Process each file
file_count = 0
for filename in os.listdir(directory):
    filepath = os.path.join(directory, filename)
    if not os.path.isfile(filepath):
        continue

    word_vector = np.zeros(len(word_list), dtype=int)

    with open(filepath, "r", encoding='utf-8', errors='ignore') as file:
        for word in file.read().split():
            word = word.lower().strip(string.punctuation)
            if word in stop_words or len(word) <= 2 or word.isdigit():
                continue
            word = lmtzr.lemmatize(word)
            if word in word_list:
                index = word_list.index(word)
                word_vector[index] += 1

    # Assign output label based on file name length (same logic preserved)
    if len(filepath) == 68:
        label = -1
    elif len(filepath) == 71:
        label = 1
    else:
        label = 0  # default/fallback label if needed

    # Write the vector and label to CSV
    with open("frequency.csv", "a", encoding='utf-8') as f_out:
        f_out.write(','.join(map(str, word_vector)) + f',{label}\n')

    file_count += 1
    if file_count % 100 == 0:
        print(f"Processed {file_count} files")

# Done
print("Time (in seconds) to segregate entire dataset to form input vector:", round(time() - start_time, 2))

Processed 100 files
Processed 200 files
Processed 300 files
Processed 400 files
Processed 500 files
Processed 600 files
Processed 700 files
Processed 800 files
Processed 900 files
Processed 1000 files
Processed 1100 files
Processed 1200 files
Processed 1300 files
Processed 1400 files
Processed 1500 files
Processed 1600 files
Processed 1700 files
Processed 1800 files
Processed 1900 files
Processed 2000 files
Processed 2100 files
Processed 2200 files
Processed 2300 files
Processed 2400 files
Processed 2500 files
Processed 2600 files
Processed 2700 files
Processed 2800 files
Processed 2900 files
Processed 3000 files
Processed 3100 files
Processed 3200 files
Processed 3300 files
Processed 3400 files
Processed 3500 files
Processed 3600 files
Processed 3700 files
Processed 3800 files
Processed 3900 files
Processed 4000 files
Processed 4100 files
Processed 4200 files
Processed 4300 files
Processed 4400 files
Processed 4500 files
Processed 4600 files
Processed 4700 files
Processed 4800 files
P

In [6]:
import os
import string
import numpy as np
import pandas as pd
from time import time
from nltk.corpus import stopwords
from nltk.stem.wordnet import WordNetLemmatizer

# Start timer
start_time = time()

# Load words from previously created wordslist.csv
df = pd.read_csv('wordslist.csv')
words = df['word']
word_list = words.tolist()  # Convert to regular list for efficiency
# Convert all elements in word_list to strings
word_list = [str(word) for word in word_list] #This line is added to convert all elements in word_list to strings using list comprehension.


# Initialize lemmatizer and stopwords
lmtzr = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# Define directory containing emails
directory = "emails/emails"  # Update if necessary

# Prepare the output CSV with header
with open("frequency.csv", "w", encoding='utf-8') as f_out:
    f_out.write(','.join(word_list) + ',output\n')

# Process each file
file_count = 0
for filename in os.listdir(directory):
    filepath = os.path.join(directory, filename)
    if not os.path.isfile(filepath):
        continue

    word_vector = np.zeros(len(word_list), dtype=int)

    with open(filepath, "r", encoding='utf-8', errors='ignore') as file:
        for word in file.read().split():
            word = word.lower().strip(string.punctuation)
            if word in stop_words or len(word) <= 2 or word.isdigit():
                continue
            word = lmtzr.lemmatize(word)
            if word in word_list:
                index = word_list.index(word)
                word_vector[index] += 1

    # Assign output label based on file name (Assuming 'spam' in filename indicates spam)
    if 'spam' in filename.lower():
        label = -1  # Spam
    else:
        label = 1   # Ham


    # Write the vector and label to CSV
    with open("frequency.csv", "a", encoding='utf-8') as f_out:
        f_out.write(','.join(map(str, word_vector)) + f',{label}\n')

    file_count += 1
    if file_count % 100 == 0:
        print(f"Processed {file_count} files")

# Done
print("Time (in seconds) to segregate entire dataset to form input vector:", round(time() - start_time, 2))

Processed 100 files
Processed 200 files
Processed 300 files
Processed 400 files
Processed 500 files
Processed 600 files
Processed 700 files
Processed 800 files
Processed 900 files
Processed 1000 files
Processed 1100 files
Processed 1200 files
Processed 1300 files
Processed 1400 files
Processed 1500 files
Processed 1600 files
Processed 1700 files
Processed 1800 files
Processed 1900 files
Processed 2000 files
Processed 2100 files
Processed 2200 files
Processed 2300 files
Processed 2400 files
Processed 2500 files
Processed 2600 files
Processed 2700 files
Processed 2800 files
Processed 2900 files
Processed 3000 files
Processed 3100 files
Processed 3200 files
Processed 3300 files
Processed 3400 files
Processed 3500 files
Processed 3600 files
Processed 3700 files
Processed 3800 files
Processed 3900 files
Processed 4000 files
Processed 4100 files
Processed 4200 files
Processed 4300 files
Processed 4400 files
Processed 4500 files
Processed 4600 files
Processed 4700 files
Processed 4800 files
P

In [7]:
import pandas as pd
import numpy as np
from time import time
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, precision_score, recall_score
from sklearn.model_selection import train_test_split
from prettytable import PrettyTable

# Load Data
df1 = pd.read_csv('wordslist.csv')  # assuming this is for reference
df2 = pd.read_csv('frequency.csv')

X = df2.iloc[:, :-1].values
Y = df2.iloc[:, -1].values.ravel()  # flatten to 1D array

# Split data (70% train, 30% test)
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=42)

# Reset results file
with open("results.txt", "w") as f:
    pass

def evaluate_model(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[1, -1])
    table = PrettyTable([' ', 'Ham (1)', 'Spam (-1)'])
    table.add_row(['Ham (1)', cm[0][0], cm[0][1]])
    table.add_row(['Spam (-1)', cm[1][0], cm[1][1]])

    precision = precision_score(y_true, y_pred, pos_label=1)
    recall = recall_score(y_true, y_pred, pos_label=1)
    return table, precision, recall

def train_and_evaluate_model(X_train, Y_train, X_test, Y_test, kernel, params, label):
    start = time()
    model = SVC(kernel=kernel, C=0.1, **params)
    model.fit(X_train, Y_train)
    predictions = model.predict(X_test)
    table, precision, recall = evaluate_model(Y_test, predictions)

    with open("results.txt", "a") as f:
        f.write(f"{label} Kernel\n")
        if kernel == 'poly':
            f.write(f"Degree: {params.get('degree', 'N/A')}, Coef0: {params.get('coef0', 'N/A')}\n")
        f.write(table.get_string())
        f.write("\n")
        f.write(f"Precision: {round(precision, 2)}\n")
        f.write(f"Recall: {round(recall, 2)}\n")
        f.write(f"Time taken: {round(time() - start, 2)}s\n\n")

# Track total time
total_start = time()

# Poly kernel variations
for degree in range(2, 4):
    for offset in range(10):
        params = {'degree': degree, 'coef0': offset}
        train_and_evaluate_model(X_train, Y_train, X_test, Y_test, kernel='poly', params=params, label='Polynomial')

# Linear kernel
train_and_evaluate_model(X_train, Y_train, X_test, Y_test, kernel='linear', params={}, label='Linear')

# Log total time
with open("results.txt", "a") as f:
    f.write(f"Total time for all models: {round(time() - total_start, 2)}s\n")


In [8]:
import pandas as pd
import numpy as np
from time import time
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, precision_score, recall_score
from sklearn.model_selection import train_test_split
from prettytable import PrettyTable

# Load Data
df1 = pd.read_csv('wordslist.csv')  # assuming this is for reference
df2 = pd.read_csv('frequency.csv')

X = df2.iloc[:, :-1].values
Y = df2.iloc[:, -1].values.ravel()  # flatten to 1D array

# Split data (70% train, 30% test)
X_train, X_test, Y_train, Y_test, indices_train, indices_test = train_test_split(
    X, Y, df2.index, test_size=0.3, random_state=42)

# Reset results file
with open("results.txt", "w") as f:
    pass

def evaluate_model(y_true, y_pred, indices):
    cm = confusion_matrix(y_true, y_pred, labels=[1, -1])
    table = PrettyTable([' ', 'Ham (1)', 'Spam (-1)'])
    table.add_row(['Ham (1)', cm[0][0], cm[0][1]])
    table.add_row(['Spam (-1)', cm[1][0], cm[1][1]])

    precision = precision_score(y_true, y_pred, pos_label=1)
    recall = recall_score(y_true, y_pred, pos_label=1)

    # Create classification details
    classification = pd.DataFrame({
        'Email Index': indices,
        'Actual': y_true,
        'Predicted': y_pred,
        'Status': ['Correct' if true == pred else 'Incorrect'
                 for true, pred in zip(y_true, y_pred)]
    })

    return table, precision, recall, classification

def train_and_evaluate_model(X_train, Y_train, X_test, Y_test, indices_test, kernel, params, label):
    start = time()
    model = SVC(kernel=kernel, C=0.1, **params)
    model.fit(X_train, Y_train)
    predictions = model.predict(X_test)
    table, precision, recall, classification = evaluate_model(Y_test, predictions, indices_test)

    # Print to console
    print(f"\n{label} Kernel")
    if kernel == 'poly':
        print(f"Degree: {params.get('degree', 'N/A')}, Coef0: {params.get('coef0', 'N/A')}")
    print(table)
    print(f"Precision: {round(precision, 2)}")
    print(f"Recall: {round(recall, 2)}")
    print(f"Time taken: {round(time() - start, 2)}s")

    # Print classification results
    print("\nClassification Results:")
    print(classification.to_string(index=False))

    # Print some examples
    print("\nSample Ham Emails (Predicted as Ham):")
    print(classification[(classification['Predicted'] == 1) & (classification['Actual'] == 1)]
              .head(5).to_string(index=False))

    print("\nSample Spam Emails (Predicted as Spam):")
    print(classification[(classification['Predicted'] == -1) & (classification['Actual'] == -1)]
              .head(5).to_string(index=False))

    print("\nFalse Positives (Ham predicted as Spam):")
    print(classification[(classification['Predicted'] == -1) & (classification['Actual'] == 1)]
              .head(5).to_string(index=False))

    print("\nFalse Negatives (Spam predicted as Ham):")
    print(classification[(classification['Predicted'] == 1) & (classification['Actual'] == -1)]
              .head(5).to_string(index=False))

    # Write to file
    with open("results.txt", "a") as f:
        f.write(f"{label} Kernel\n")
        if kernel == 'poly':
            f.write(f"Degree: {params.get('degree', 'N/A')}, Coef0: {params.get('coef0', 'N/A')}\n")
        f.write(table.get_string())
        f.write("\n")
        f.write(f"Precision: {round(precision, 2)}\n")
        f.write(f"Recall: {round(recall, 2)}\n")
        f.write(f"Time taken: {round(time() - start, 2)}s\n\n")

# Track total time
total_start = time()

# Poly kernel variations
print("Evaluating Polynomial Kernels...")
for degree in range(2, 4):
    for offset in range(10):
        params = {'degree': degree, 'coef0': offset}
        train_and_evaluate_model(X_train, Y_train, X_test, Y_test, indices_test,
                               kernel='poly', params=params, label='Polynomial')

# Linear kernel
print("\nEvaluating Linear Kernel...")
train_and_evaluate_model(X_train, Y_train, X_test, Y_test, indices_test,
                       kernel='linear', params={}, label='Linear')

# Log total time
total_time = round(time() - total_start, 2)
print(f"\nTotal time for all models: {total_time}s")
with open("results.txt", "a") as f:
    f.write(f"Total time for all models: {total_time}s\n")

Streaming output truncated to the last 5000 lines.
        3378       1          1   Correct
        9795      -1         -1   Correct
        5060       1          1   Correct
        9647      -1         -1   Correct
        5767      -1          1 Incorrect
        3434       1          1   Correct
        6615      -1          1 Incorrect
        5799       1          1   Correct
         461       1          1   Correct
        9074      -1          1 Incorrect
        7985       1          1   Correct
        1360       1          1   Correct
        9875       1          1   Correct
        1175      -1         -1   Correct
         376      -1          1 Incorrect
         410       1          1   Correct
        8850       1          1   Correct
        3656      -1          1 Incorrect
        5558       1          1   Correct
        5749       1          1   Correct
        1740       1          1   Correct
        9066       1          1   Correct
        7432       1     

In [22]:
import pandas as pd
import numpy as np

# Load your data (replace with actual file path if it's a CSV/Excel)
df = pd.read_csv('Resume.csv')  # or pd.read_excel('resume_data.xlsx')
df.drop(columns=["Recruiter Decision", "AI Score (0-100)"], inplace=True)


# Normalize skill names
def normalize_skills(skill_str):
    if pd.isnull(skill_str):
        return []
    skills = [s.strip().lower() for s in skill_str.split(',')]
    return list(set(skills))  # remove duplicates

df['Skills'] = df['Skills'].apply(normalize_skills)

# Handle missing or "None" certifications
df['Certifications'] = df['Certifications'].replace(['None', 'none', '', np.nan], 'No Certification')

# Optional: Normalize Education field (e.g., make consistent casing)
df['Education'] = df['Education'].str.strip().str.upper()

# Show cleaned data
df.head()


,Resume_ID,Name,Skills,Experience (Years),Education,Certifications,Job Role,Salary Expectation ($),Projects Count,Email Address
0,1,Ashley Ali,"[pytorch, tensorflow, nlp]",10,B.SC,No Certification,AI Researcher,104895,8,ashley.ali993@gmail.com
1,2,Wesley Roman,"[python, deep learning, sql, machine learning]",10,MBA,Google ML,Data Scientist,113002,1,wesley.roman323@fastmail.com
2,3,Corey Sanchez,"[linux, cybersecurity, ethical hacking]",1,MBA,Deep Learning Specialization,Cybersecurity Analyst,71766,7,corey.sanchez414@outlook.com
3,4,Elizabeth Carney,"[python, pytorch, tensorflow]",7,B.TECH,AWS Certified,AI Researcher,46848,0,elizabeth.carney986@fastmail.com
4,5,Julie Hill,"[java, sql, react]",4,PHD,No Certification,Software Engineer,87441,9,julie.hill824@gmail.com


In [23]:
!pip install plotly


In [24]:
import plotly.express as px
import pandas as pd


# 1. Salary vs Experience (Scatter Plot)
fig1 = px.scatter(
    df,
    x="Experience (Years)",
    y="Salary Expectation ($)",
    color="Job Role",
    hover_data=["Name", "Education", "Certifications", "Projects Count"],
    title="Salary Expectation vs Experience",
    template="plotly_dark"
)
fig1.show()

# 2. Job Role Distribution (Pie Chart)
fig2 = px.pie(
    df,
    names="Job Role",
    title="Distribution of Candidates by Job Role",
    hole=0.3,
    template="plotly_dark"
)
fig2.show()


In [ ]:
def recommend_candidates(df, desired_skill, top_n=5):
    # Normalize the skill input
    desired_skill = desired_skill.lower().strip()

    recommendations = []

    for _, row in df.iterrows():
        candidate_skills = [skill.lower().strip() for skill in row["Skills"]]
        skill_match = desired_skill in candidate_skills

        # Score based on skill match and additional factors
        score = 0
        if skill_match:
            score += 10
        score += row["Experience (Years)"] * 0.5
        score += row["Projects Count"] * 0.3
        score += 2 if row["Certifications"] != "None" else 0

        recommendations.append((row["Name"], row["Job Role"], row["Skills"], row["Certifications"],
                                row["Experience (Years)"], row["Projects Count"], score))

    # Sort by score
    sorted_recommendations = sorted(recommendations, key=lambda x: x[-1], reverse=True)

    # Convert to DataFrame
    rec_df = pd.DataFrame(sorted_recommendations[:top_n],
                          columns=["Name", "Job Role", "Skills", "Certifications",
                                   "Experience (Years)", "Projects Count", "Score"])

    return rec_df


In [30]:
import plotly.express as px
import plotly.graph_objects as go

def advanced_recommendation(df, desired_skills, min_experience=0, cert_required=None, top_n=5):
    desired_skills = [skill.lower().strip() for skill in desired_skills]

    recommendations = []

    for _, row in df.iterrows():
        candidate_skills = [s.lower().strip() for s in row["Skills"]]
        skill_match_count = len(set(desired_skills).intersection(candidate_skills))
        skill_coverage = skill_match_count / len(desired_skills) if desired_skills else 0

        experience_score = np.clip(row["Experience (Years)"] / 10, 0, 1)
        project_score = np.clip(row["Projects Count"] / 10, 0, 1)
        cert_score = 1 if cert_required and cert_required.lower() in row["Certifications"].lower() else 0

        total_score = (skill_coverage * 0.5) + (experience_score * 0.2) + (project_score * 0.2) + (cert_score * 0.1)

        if row["Experience (Years)"] >= min_experience:
            recommendations.append({
                "Name": row["Name"],
                "Job Role": row["Job Role"],
                "Skills": row["Skills"],
                "Certifications": row["Certifications"],
                "Experience (Years)": row["Experience (Years)"],
                "Projects Count": row["Projects Count"],
                "Score": round(total_score, 3),
                "Salary Expectation ($)": row["Salary Expectation ($)"]
            })

    top_candidates = sorted(recommendations, key=lambda x: x["Score"], reverse=True)[:top_n]
    return pd.DataFrame(top_candidates)

print(advanced_recommendation(
    df,
    desired_skills=["Python", "TensorFlow", "NLP"],
    min_experience=5,
    cert_required="AWS"
))



# Get top candidates from recommendation system
top_candidates_df = advanced_recommendation(
    df,
    desired_skills=["Python", "TensorFlow", "NLP"],
    min_experience=5,
    cert_required="AWS"
)

best_candidate = top_candidates_df.sort_values(by="Score", ascending=False).iloc[0]

from prettytable import PrettyTable

def print_top_candidates(df, top_n=5):
    table = PrettyTable()
    table.field_names = df.columns.tolist()

    for idx in range(min(top_n, len(df))):
        table.add_row(df.iloc[idx].tolist())

    print(f"💼 Top {top_n} Recommended Candidates:\n")
    print(table)

# Call the function to print top 5
print_top_candidates(top_candidates_df, top_n=5)







             Name       Job Role                              Skills  \
0   Timothy Smith  AI Researcher  [python, pytorch, tensorflow, nlp]   
1  Christine Beck  AI Researcher  [python, pytorch, tensorflow, nlp]   
2     Dana Gibson  AI Researcher  [python, pytorch, tensorflow, nlp]   
3  Chelsea Harris  AI Researcher           [python, tensorflow, nlp]   
4     John Wilson  AI Researcher  [python, pytorch, tensorflow, nlp]   

  Certifications  Experience (Years)  Projects Count  Score  \
0  AWS Certified                   9              10   0.98   
1  AWS Certified                   9              10   0.98   
2  AWS Certified                   8              10   0.96   
3  AWS Certified                   7              10   0.94   
4  AWS Certified                   7              10   0.94   

   Salary Expectation ($)  
0                   95912  
1                  112005  
2                   44783  
3                   49447  
4                  112353  
💼 Top 5 Recommended 

In [31]:
fig1 = px.bar(
    top_candidates_df,
    x="Name",
    y="Score",
    color="Job Role",
    text="Score",
    title="Top Recommended Candidates by Score",
    template="plotly_dark"

)
fig1.update_layout(yaxis_title="Match Score", xaxis_title="Candidate")
fig1.show()

fig2 = px.scatter(
    top_candidates_df,
    x="Experience (Years)",
    y="Salary Expectation ($)",
    text="Name",
    color="Job Role",
    size="Score",
    title="Salary vs Experience for Top Candidates",
    template="plotly_dark"
)
fig2.update_traces(textposition='top center')
fig2.show()